In [1]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, chi2

# Load Dataset
dataset = pd.read_csv("prep.csv")
df = pd.get_dummies(dataset, drop_first=True)
X = df.drop(columns=["classification_yes"])
y = df["classification_yes"]

# K Values
k_values = list(range(1, 11))
print("K values:", k_values)


K values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [2]:
# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

models = {
    "Linear": LinearRegression(),
    "SVM Linear": SVR(kernel="linear"),
    "SVM RBF": SVR(kernel="rbf"),
    "Decision Tree": DecisionTreeRegressor(random_state=0),
    "Random Forest": RandomForestRegressor(n_estimators=10, random_state=0)
}


In [3]:
# SelectKBest Chi-Square for K = 1 to 10
# Target and method are retained from the original regression .py file
results = []
selected_features = []

for k in k_values:
    selector = SelectKBest(score_func=chi2, k=k)
    X_k = selector.fit_transform(X, y)
    features = X.columns[selector.get_support()].tolist()
    selected_features.append({"K": k, "Selected Features": ", ".join(features)})

    X_train, X_test, y_train, y_test = train_test_split(
        X_k, y, test_size=0.25, random_state=0
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_test_scaled)
        r2 = r2_score(y_test, pred)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        results.append({"K": k, "Model": name, "R2": r2, "R2 %": r2*100, "RMSE": rmse})

results_df = pd.DataFrame(results)
r2_table = results_df.pivot(index="K", columns="Model", values="R2 %")
r2_table.round(2)


Model,Decision Tree,Linear,Random Forest,SVM Linear,SVM RBF
K,,,,,
1,-16.98,6.16,-9.21,-29.34,-10.59
2,-3.08,21.88,15.59,-11.12,28.76
3,26.22,28.80,52.82,25.51,33.35
4,47.92,30.50,59.94,25.69,43.08
5,69.62,55.20,83.68,54.54,74.97
6,86.98,59.90,89.76,58.64,83.90
7,82.64,65.70,91.62,64.19,89.30
8,86.98,64.65,89.89,61.22,89.13
9,86.98,64.61,91.97,60.25,90.18


In [4]:
# RMSE Table
rmse_table = results_df.pivot(index="K", columns="Model", values="RMSE")
rmse_table.round(4)


Model,Decision Tree,Linear,Random Forest,SVM Linear,SVM RBF
K,,,,,
1,0.5191,0.4650,0.5016,0.5459,0.5048
2,0.4873,0.4242,0.4410,0.5060,0.4052
3,0.4123,0.4050,0.3297,0.4143,0.3919
4,0.3464,0.4002,0.3038,0.4138,0.3621
5,0.2646,0.3213,0.1939,0.3236,0.2402
6,0.1732,0.3039,0.1536,0.3087,0.1926
7,0.2000,0.2811,0.1389,0.2872,0.1570
8,0.1732,0.2854,0.1526,0.2989,0.1583
9,0.1732,0.2855,0.1360,0.3026,0.1504


In [5]:
# Selected Features for Each K
selected_features_df = pd.DataFrame(selected_features)
selected_features_df


,K,Selected Features
0,1,wc
1,2,"bgr, wc"
2,3,"bgr, bu, wc"
3,4,"bgr, bu, sc, wc"
4,5,"bgr, bu, sc, pcv, wc"
5,6,"al, bgr, bu, sc, pcv, wc"
6,7,"al, bgr, bu, sc, hrmo, pcv, wc"
7,8,"age, al, bgr, bu, sc, hrmo, pcv, wc"
8,9,"age, al, su, bgr, bu, sc, hrmo, pcv, wc"
9,10,"age, al, su, bgr, bu, sc, hrmo, pcv, wc, htn_yes"


In [6]:
# Best K and Best Regression Model
best = results_df.loc[results_df["R2"].idxmax()]
print("Best K:", int(best["K"]))
print("Best Model:", best["Model"])
print("Best R2:", round(best["R2 %"], 2), "%")
print("Best RMSE:", round(best["RMSE"], 4))
print("Reason: Highest R2 score among K = 1 to 10; RMSE is also shown for error comparison.")


Best K: 10
Best Model: Random Forest
Best R2: 92.27 %
Best RMSE: 0.1334
Reason: Highest R2 score among K = 1 to 10; RMSE is also shown for error comparison.
